# GAN 네 조건 FID·LPIPS 예비 점검 — ZIP 직접 업로드

Drive 연결 없이 기존 **`gan_four_conditions_20261101_run01_results.zip`** 결과 ZIP을 직접 업로드하여 분석합니다. 4조건의 GAN 학습에 사용된 실제 사진 각 50장과 생성 사진 각 50장을 대상으로 합니다. GAN·분류기 학습은 진행하지 않습니다.

## 실행 전
1. Colab에서 **런타임 → 런타임 유형 변경 → G4 GPU**를 선택하세요. 업로드 후 런타임을 변경하면 업로드 파일이 사라지므로 먼저 선택해야 합니다. 업로드 대기 중에도 G4가 연결되어 있으면 사용 시간이 발생할 수 있습니다.
2. ZIP을 `Downloads` 등에 준비하고 아래 1번 셀을 실행해 선택합니다. Drive 마운트는 사용하지 않습니다.
3. 2번 셀은 `torchmetrics`, `torch-fidelity`, `lpips` 설치와 사전학습 Inception·AlexNet 가중치 다운로드가 필요합니다. 설치·다운로드가 실패하면 오류가 출력됩니다.
4. 3번 셀에서 **결과 ZIP 다운로드가 시작되고 30초 뒤 런타임이 자동 반납**됩니다. 30초가 다운로드 완료를 보장하지 않으니 컴퓨터에서 ZIP 수신 여부를 확인하세요. 점검 수치는 2번 셀에도 출력됩니다. 연결이 끊긴 경우에는 다운로드가 실패할 수 있습니다.

## 해석 주의
- FID: 실제 GAN 학습 사진 50장과 생성 50장의 특징 분포 차이(작을수록 유사). 표본이 작고 실제 참조 집합이 달라 예비 참고값입니다.
- LPIPS: 생성 사진끼리 100쌍의 거리와 생성 사진별 실제 GAN 학습 사진 중 가장 가까운 거리를 확인합니다. 무작위 실제·생성을 정답 짝으로 간주하지 않습니다. 거리만으로 이미지가 유용하거나 복제됐다고 단정할 수 없습니다.
- 이번 실제 참조에는 과거 검증·평가 구간 사진이 일부 포함됩니다. 독립 검증이 아니며, 후속 성능 실험은 학습 구간에서 GAN을 다시 학습하거나 새로운 독립 출력으로 평가해야 합니다. 클래스별 원래 출력은 각 1회입니다.


## 1. 기존 결과 ZIP 업로드

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, zipfile, hashlib

print('기존 GAN 결과 ZIP 파일을 선택하세요.',flush=True)
uploads = files.upload()
if len(uploads) != 1:
    raise ValueError('결과 ZIP 파일을 정확히 하나만 선택하세요.')
upload_name, upload_bytes = next(iter(uploads.items()))
if not upload_name.lower().endswith('.zip'):
    raise ValueError('ZIP 파일을 선택하세요.')
upload_sha256 = hashlib.sha256(upload_bytes).hexdigest()
source = Path('/content/gan_quality_manual_source_20261101')
if source.exists() and any(source.iterdir()):
    raise RuntimeError('기존 업로드가 있습니다. 런타임을 새로 시작한 후 다시 업로드하세요.')
source.mkdir(parents=True,exist_ok=True)
groups = ('clean_normal','clean_underfill','partial_normal','partial_underfill')
allowed = tuple([f'prepared/data/{g}/' for g in groups] + [f'gan/{g}/generated/' for g in groups])
with zipfile.ZipFile(io.BytesIO(upload_bytes)) as z:
    bad = z.testzip()
    if bad: raise ValueError(f'ZIP 손상: {bad}')
    for item in z.infolist():
        name = item.filename
        if item.is_dir() or not name.endswith('.png') or not name.startswith(allowed):
            continue
        parts = PurePosixPath(name).parts
        if not parts or name.startswith('/') or '\\' in name or any(p in ('..','') for p in parts):
            raise ValueError(f'잘못된 ZIP 경로: {name}')
        dest = source.joinpath(*parts)
        dest.parent.mkdir(parents=True,exist_ok=True)
        dest.write_bytes(z.read(item))
for g in groups:
    real = len(list((source/'prepared'/'data'/g).glob('*.png')))
    fake = len(list((source/'gan'/g/'generated').glob('*.png')))
    print(f'{g}: GAN 학습 실제 {real}장, 생성 {fake}장',flush=True)
    if real != 50 or fake != 50:
        raise ValueError('네 조건 각각 실제·생성 50장이 필요합니다. 선택한 ZIP을 확인하세요.')
del upload_bytes, uploads
print('업로드 및 입력 확인 완료. ZIP SHA-256:',upload_sha256,flush=True)


## 2. FID·LPIPS 점검

동일한 RGB 256×256 입력, 생성 간 100쌍 및 생성별 학습 사진 최근접 거리를 사용합니다. 결과 수치는 아래 셀 출력과 CSV에 저장됩니다.

In [ ]:
from pathlib import Path
import sys, subprocess, traceback, json, os, time, hashlib, random, csv, zipfile, itertools, tempfile
if 'source' not in globals() or not source.is_dir():
    raise RuntimeError('먼저 1번 ZIP 업로드 셀을 실행하세요.')
run_root = Path(tempfile.mkdtemp(prefix='gan_quality_metrics_',dir='/content'))
try:
    log_dir = run_root/'logs'; log_dir.mkdir()
    install_log = log_dir / 'install.log'
    print('평가 의존 패키지를 설치합니다. 기존 학습 결과는 변경하지 않습니다.', flush=True)
    cmd = [sys.executable, '-m', 'pip', 'install', 'torchmetrics==1.8.2', 'torch-fidelity==0.3.0', 'lpips==0.1.4']
    with install_log.open('w', encoding='utf-8') as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
        for line in p.stdout:
            f.write(line); f.flush()
        if p.wait() != 0:
            raise RuntimeError(f'패키지 설치 실패: {install_log}')
    import numpy as np
    import torch
    import torchvision
    from PIL import Image, ImageOps, ImageDraw
    import lpips
    import torchmetrics
    from torchmetrics.image.fid import FrechetInceptionDistance

    if not torch.cuda.is_available():
        raise RuntimeError('GPU가 없습니다. G4를 선택하세요.')
    gpu = torch.cuda.get_device_name(0)
    if 'RTX PRO 6000' not in gpu.upper():
        raise RuntimeError(f'선택한 GPU가 G4가 아닙니다: {gpu}')
    device = torch.device('cuda')
    print('GPU:', gpu, flush=True)
    seed = 42
    groups = ['clean_normal', 'clean_underfill', 'partial_normal', 'partial_underfill']
    settings = {'seed': seed, 'gpu': gpu, 'source': str(source), 'n_real': 50, 'n_generated': 50,
                'image_format': 'RGB PNG 256x256; real and generated equally loaded',
                'fid': 'torchmetrics FrechetInceptionDistance feature=2048 normalize=True (torch-fidelity Inception)',
                'lpips': 'lpips 0.1.4 alex pretrained, RGB float [-1,1]',
                'lpips_generated_pairs': '100 distinct unordered pairs sampled without replacement seed=42 separately per group',
                'lpips_nearest': 'each generated image against all 50 GAN training images, minimum',
                'limitations': 'Training real reference; 50 images; some prior held-out frames included; single print per class; preliminary only.',
                'input_zip_sha256':upload_sha256, 'input_zip_name':upload_name,
                'versions': {'torch':torch.__version__, 'torchvision':torchvision.__version__, 'torchmetrics':torchmetrics.__version__, 'numpy':np.__version__}}
    (run_root/'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
    summary_rows, pair_rows, nearest_rows = [], [], []
    review_dir = run_root/'review'; review_dir.mkdir()

    def sha(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
        return h.hexdigest()

    def load_images(paths):
        samples = []
        for path in paths:
            with Image.open(path) as im:
                if im.size != (256, 256) or im.mode != 'RGB':
                    raise ValueError(f'예상과 다른 입력: {path} / {im.mode} / {im.size}')
                a = np.asarray(im, dtype=np.uint8).copy()
            samples.append(torch.from_numpy(a).permute(2,0,1))
        return torch.stack(samples)

    # LPIPS의 AlexNet과 보정 가중치를 다운로드할 수 있습니다.
    lp_model = lpips.LPIPS(net='alex', verbose=False).eval().to(device)
    for param in lp_model.parameters(): param.requires_grad_(False)
    input_hashes = []
    for group in groups:
        print(f'[{group}] 입력 확인 및 점검 중', flush=True)
        real_dir = source/'prepared'/'data'/group
        fake_dir = source/'gan'/group/'generated'
        real_files, fake_files = sorted(real_dir.glob('*.png')), sorted(fake_dir.glob('*.png'))
        if len(real_files) != 50 or len(fake_files) != 50:
            raise RuntimeError(f'{group}: 실사 {len(real_files)}장, 생성 {len(fake_files)}장. 각 50장 필요')
        for kind, paths in (('real_train',real_files),('generated',fake_files)):
            input_hashes.extend({'group':group,'type':kind,'filename':p.name,'sha256':sha(p)} for p in paths)
        real = load_images(real_files)
        fake = load_images(fake_files)
        # 표준 FID: 두 묶음에 같은 입력 규칙을 적용합니다.
        fid_score, fid_status = None, 'ok'
        try:
            fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
            for start in range(0,50,16):
                fid.update(real[start:start+16].to(device).float()/255, real=True)
                fid.update(fake[start:start+16].to(device).float()/255, real=False)
            with torch.no_grad(): fid_value = float(fid.compute().detach().cpu())
            if not np.isfinite(fid_value): raise ValueError(f'FID가 유한하지 않습니다: {fid_value}')
            fid_score = fid_value
            del fid
        except Exception as exc:
            fid_status = f'failed: {type(exc).__name__}: {exc}'
            print('FID 계산 실패:', fid_status, flush=True)
        # LPIPS용 범위 [-1,1]. 생성 100쌍과 50x50 최근접 이미지 검색.
        rr = real.to(device).float()/127.5-1.0
        ff = fake.to(device).float()/127.5-1.0
        pairs = random.Random(seed).sample(list(itertools.combinations(range(50),2)),100)
        pair_values = []
        with torch.no_grad():
            for start in range(0,100,16):
                p = pairs[start:start+16]
                a = torch.tensor([x for x,y in p], device=device)
                b = torch.tensor([y for x,y in p], device=device)
                vals = lp_model(ff[a],ff[b]).flatten().detach().cpu().tolist()
                for (i,j), value in zip(p,vals):
                    if not np.isfinite(value): raise ValueError('LPIPS 생성 쌍 값이 유한하지 않음')
                    pair_values.append(float(value))
                    pair_rows.append({'group':group,'generated_a':fake_files[i].name,'generated_b':fake_files[j].name,'lpips':value})
            group_nearest=[]
            for i in range(50):
                vals=[]
                for start in range(0,50,16):
                    chunk=rr[start:start+16]
                    distances=lp_model(ff[i:i+1].expand(len(chunk),-1,-1,-1),chunk).flatten()
                    vals.extend(distances.detach().cpu().tolist())
                if not np.all(np.isfinite(vals)): raise ValueError('LPIPS 최근접 값이 유한하지 않음')
                j=int(np.argmin(vals))
                item={'group':group,'generated':fake_files[i].name,'nearest_training_real':real_files[j].name,'lpips':float(vals[j])}
                group_nearest.append(item); nearest_rows.append(item)
        # 가까운 순으로 최대 8쌍. 후보일 뿐 복제 판정이 아닙니다.
        examples=sorted(group_nearest,key=lambda x:x['lpips'])[:8]
        tile, gap, header = 160, 10, 32
        board=Image.new('RGB',(4*(2*tile+gap)+3*gap,2*(tile+header)+gap),'white')
        draw=ImageDraw.Draw(board)
        for k,item in enumerate(examples):
            x=(k%4)*(2*tile+2*gap); y=(k//4)*(tile+header+gap)
            with Image.open(fake_dir/item['generated']) as im:
                board.paste(ImageOps.fit(im.convert('RGB'),(tile,tile)),(x,y))
            with Image.open(real_dir/item['nearest_training_real']) as im:
                board.paste(ImageOps.fit(im.convert('RGB'),(tile,tile)),(x+tile+gap,y))
            draw.text((x,y+tile+3),f"{k+1}: generated | train, LPIPS={item['lpips']:.4f}",fill='black')
        board.save(review/f'{group}_nearest_8.jpg',quality=94)
        nearest_values=[x['lpips'] for x in group_nearest]
        summary_rows.append({'group':group,'n_train_real':50,'n_generated':50,'FID_2048':fid_score,'FID_status':fid_status,
                             'LPIPS_100_generated_pairs_mean':float(np.mean(pair_values)),
                             'LPIPS_100_generated_pairs_median':float(np.median(pair_values)),
                             'LPIPS_nearest_train_mean':float(np.mean(nearest_values)),
                             'LPIPS_nearest_train_median':float(np.median(nearest_values)),
                             'LPIPS_nearest_train_min':float(np.min(nearest_values))})
        print(f'[{group}] 완료. FID 상태: {fid_status}; LPIPS 이미지 검토 필요', flush=True)
        del rr,ff,real,fake
        torch.cuda.empty_cache()
    def writecsv(name, rows):
        with (run_root/name).open('w',encoding='utf-8-sig',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    writecsv('summary.csv',summary_rows);writecsv('lpips_generated_pairs.csv',pair_rows)
    writecsv('lpips_nearest_training.csv',nearest_rows);writecsv('input_sha256.csv',input_hashes)
    print('조건별 예비 수치:',json.dumps(summary_rows,ensure_ascii=False,indent=2),flush=True)
    (run_root/'README_결과_읽기.md').write_text("""# 읽는 방법

summary.csv의 FID는 실제 GAN 학습 사진 50장과 생성 50장의 표준 FID 예비값이며 작을수록 특징 분포가 가깝습니다. 표본이 작고 서로 다른 조건의 실제 기준 사진이 달라 조건 간 숫자로 성능 순위를 주장할 수 없습니다.

생성 이미지 간 LPIPS 100쌍은 다양성 참고값이고, 생성별 학습 사진 최근접 LPIPS는 복제 의심 여부를 육안 확인할 후보를 제공합니다. 노이즈나 마스킹이 점수에 영향을 줄 수 있으며 어떤 거리도 복제 여부나 좋은 품질의 증거는 아닙니다. review 이미지는 가까운 순 8쌍이며 왼쪽 생성, 오른쪽 실제 학습 사진입니다.

이번 GAN은 기존 검증·평가 구간 사진도 일부 포함하므로 생성 품질의 독립 검증 또는 분류기 성능 검증으로 사용할 수 없습니다. 후속 실험은 학습 구간에서 GAN을 다시 학습하거나 독립된 새 출력으로 평가해야 합니다. 정상과 언더필은 각각 출력 1회입니다. 입력 이미지·파일 지문과 설정·버전·랜덤 시드는 별도 파일에 보존됩니다.
""",encoding='utf-8')
    zip_path=run_root/'gan_quality_metrics_results.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in run_root.rglob('*'):
            if p.is_file() and p != zip_path: z.write(p,p.relative_to(run_root).as_posix())
    with zipfile.ZipFile(zip_path) as z:
        bad=z.testzip()
        if bad: raise RuntimeError(f'ZIP 손상: {bad}')
    print('점검 완료. 다음 다운로드 셀을 실행하세요:',zip_path,flush=True)
except BaseException as exc:
    trace=traceback.format_exc()
    print('점검 실패:',trace,flush=True)
    if run_root is not None:
        try:
            run_root.mkdir(parents=True,exist_ok=True)
            (run_root/'error.log').write_text(trace,encoding='utf-8')
        except Exception: pass
    raise


## 3. 결과 ZIP 다운로드 및 G4 반납

다운로드 시작 후 30초 뒤 반납합니다. 파일 수신 여부를 직접 확인하세요.

In [ ]:
from google.colab import files, runtime
import time
if 'zip_path' not in globals() or not zip_path.is_file():
    raise RuntimeError('결과 ZIP이 없습니다. 앞의 점검 셀 오류를 확인하세요.')
print('결과 ZIP 다운로드를 시작합니다:',zip_path.name,flush=True)
print('30초 후 G4 런타임 반납을 요청합니다. 컴퓨터에서 다운로드 완료를 확인하세요.',flush=True)
files.download(str(zip_path))
time.sleep(30)
print('G4 런타임 반납 요청',flush=True)
runtime.unassign()
